# 12 — Cross-direction comparison: S/U vs Assistant Axis vs Refusal

With four directions in Llama 3.3 70B's residual space at layer 50, this notebook answers two questions:

1. **How aligned are they?** Pairwise cosine matrix tells you if they're orthogonal axes (independent phenomena), aligned (the same axis with different names), or partially overlapping.
2. **What features does each one decompose into?** Project each direction onto Goodfire's SAE → top-K features per direction → compute set overlap (Jaccard) and rank correlation between feature lists.

The four directions (set in config below):
- **S/U**: our exp10 fit on `llama33_70b` — instruction-source authority
- **Assistant Axis**: pre-computed, from `lu-christina/assistant-axis-vectors/llama-3.3-70b/assistant_axis.pt`
- **Default Vector**: also from the assistant-axis dataset — "baseline assistant persona" centroid
- **Refusal**: from `huihui-ai/Llama-3.3-70B-Instruct-abliterated/refusal_dir.pth`

All directions live in the same `d_model=8192` residual basis, so cosine and SAE-projection are directly comparable. **No model loading required**; only the SAE (4.3 GB) and the direction files (~1 MB each).

In [ ]:
# Install (Colab / fresh box). Comment out if already in env.
# !pip install -q huggingface_hub torch numpy matplotlib seaborn

In [ ]:
# ---- Config ----
from pathlib import Path

# Layer at which to do the projection. Goodfire's SAE is fixed at L50.
SAE_LAYER       = 50

# Goodfire SAE
SAE_REPO_ID     = 'Goodfire/Llama-3.3-70B-Instruct-SAE-l50'
SAE_FILENAME    = 'Llama-3.3-70B-Instruct-SAE-l50.pt'

# Local exp10 NPZ + key (the S/U direction we fit ourselves).
SUI_NPZ_PATH    = Path.cwd().parent / 'exp06_directions_v2' / 'llama33_70b' / 'directions.npz'
SUI_KEY         = f'mm_dir__response_last__layer_{SAE_LAYER:03d}'

# Top-K features per direction
TOP_K           = 25

# Whether to include each direction in the comparison (set False to skip if file missing)
INCLUDE = {
    'sui':            True,   # our S/U direction
    'assistant_axis': True,
    'default_vector': True,
    'refusal':        True,
}

print(f'comparing {sum(INCLUDE.values())} directions at L{SAE_LAYER} of Llama 3.3 70B')

## 1 — Load all directions into a single (n_dirs, d_model) matrix

Normalizes each to unit norm. Skips any whose file is missing (with a warning) so the notebook still runs partially.

In [ ]:
import torch
import numpy as np
from huggingface_hub import hf_hub_download

DIRECTIONS = {}   # name -> (d_model,) unit tensor

def _select_layer(t: torch.Tensor, layer: int, name: str) -> torch.Tensor:
    """If t is (n_layers, d_model), pick the row for `layer`. If 1D, return as-is."""
    t = t.float().squeeze()
    if t.ndim == 2:
        print(f'  {name}: 2D tensor {tuple(t.shape)} — picking row {layer}')
        return t[layer]
    if t.ndim == 1:
        return t
    raise ValueError(f'{name}: unexpected tensor shape {tuple(t.shape)}')

def _normalize(t: torch.Tensor) -> torch.Tensor:
    return t / (t.norm() + 1e-10)

# ---- our S/U direction (local) ----
if INCLUDE['sui']:
    if SUI_NPZ_PATH.exists():
        arr = np.load(SUI_NPZ_PATH)
        if SUI_KEY in arr.files:
            DIRECTIONS['sui'] = _normalize(torch.from_numpy(arr[SUI_KEY]).float())
            print(f'sui            loaded from {SUI_NPZ_PATH.name} key={SUI_KEY!r}')
        else:
            print(f'sui            SKIP — key {SUI_KEY!r} not in NPZ; available: {arr.files[:5]}')
    else:
        print(f'sui            SKIP — no NPZ at {SUI_NPZ_PATH} (run notebook 10 first)')

# ---- assistant axis (HF) ----
if INCLUDE['assistant_axis']:
    try:
        p = hf_hub_download(
            repo_id='lu-christina/assistant-axis-vectors',
            filename='llama-3.3-70b/assistant_axis.pt',
            repo_type='dataset',
        )
        obj = torch.load(p, map_location='cpu', weights_only=False)
        DIRECTIONS['assistant_axis'] = _normalize(_select_layer(obj, SAE_LAYER, 'assistant_axis'))
        print(f'assistant_axis loaded from HF')
    except Exception as e:
        print(f'assistant_axis SKIP — {e}')

# ---- default vector (HF) ----
if INCLUDE['default_vector']:
    try:
        p = hf_hub_download(
            repo_id='lu-christina/assistant-axis-vectors',
            filename='llama-3.3-70b/default_vector.pt',
            repo_type='dataset',
        )
        obj = torch.load(p, map_location='cpu', weights_only=False)
        DIRECTIONS['default_vector'] = _normalize(_select_layer(obj, SAE_LAYER, 'default_vector'))
        print(f'default_vector loaded from HF')
    except Exception as e:
        print(f'default_vector SKIP — {e}')

# ---- refusal direction (HF abliterated) ----
if INCLUDE['refusal']:
    try:
        p = hf_hub_download(
            repo_id='huihui-ai/Llama-3.3-70B-Instruct-abliterated',
            filename='refusal_dir.pth',
        )
        obj = torch.load(p, map_location='cpu', weights_only=False)
        DIRECTIONS['refusal'] = _normalize(_select_layer(obj, SAE_LAYER, 'refusal'))
        print(f'refusal        loaded from HF')
    except Exception as e:
        print(f'refusal        SKIP — {e}')

assert DIRECTIONS, 'no directions loaded — check INCLUDE flags and file paths.'

# Stack into (n_dirs, d_model)
names = list(DIRECTIONS.keys())
M = torch.stack([DIRECTIONS[n] for n in names], dim=0)
print(f'\nstacked: {len(names)} directions × d_model={M.shape[1]}')
print(f'names: {names}')

# Sanity: all same dim?
assert all(d.shape == M[0].shape for d in DIRECTIONS.values())

## 2 — Pairwise cosine similarity

Quick read on whether the directions are orthogonal (cos ≈ 0), aligned (cos ≈ 1), or anti-aligned (cos ≈ -1). Diagonals are 1 by construction.

In [ ]:
cos_matrix = (M @ M.T).numpy()    # (n_dirs, n_dirs)

print('Pairwise cosine similarity:')
print(' ' * 18 + ' '.join(f'{n:>16s}' for n in names))
for i, n in enumerate(names):
    row = ' '.join(f'{cos_matrix[i, j]:+16.4f}' for j in range(len(names)))
    print(f'{n:>16s}  {row}')

# Plot heatmap
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(1.2 + 1.0 * len(names), 1.0 + 0.9 * len(names)))
    im = ax.imshow(cos_matrix, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f'{cos_matrix[i, j]:+.2f}', ha='center', va='center',
                    color='white' if abs(cos_matrix[i, j]) > 0.5 else 'black', fontsize=9)
    ax.set_title(f'Direction cosine (Llama 3.3 70B, L{SAE_LAYER})')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not available, skipping heatmap)')

## 3 — Load Goodfire SAE (4.3 GB)

Same loader as notebook 11. Extracts `W_enc` (d_sae, d_model).

In [ ]:
sae_path = hf_hub_download(repo_id=SAE_REPO_ID, filename=SAE_FILENAME)
sae_obj = torch.load(sae_path, map_location='cpu', weights_only=False)
sd = sae_obj.state_dict() if hasattr(sae_obj, 'state_dict') else sae_obj

def _find(state_dict, candidates):
    for c in candidates:
        if c in state_dict:
            return state_dict[c]
    raise KeyError(f'none of {candidates} found; first keys: {list(state_dict.keys())[:8]}')

W_enc = _find(sd, ['encoder.weight', 'W_enc', 'enc.weight', 'W_in']).float()
W_dec = _find(sd, ['decoder.weight', 'W_dec', 'dec.weight', 'W_out']).float()
if W_dec.shape[0] > W_dec.shape[1]:
    W_dec = W_dec.T
d_sae, d_model = W_enc.shape
print(f'W_enc: ({d_sae}, {d_model})  W_dec: {tuple(W_dec.shape)}')
assert d_model == M.shape[1], f'SAE d_model={d_model} != direction d_model={M.shape[1]}'

## 4 — Project every direction onto SAE features

Produces a `(n_dirs, d_sae)` matrix of cosine similarities.

In [ ]:
with torch.no_grad():
    enc_norms = W_enc.norm(dim=1) + 1e-10            # (d_sae,)
    feat_sims = (M @ W_enc.T) / enc_norms             # (n_dirs, d_sae)

print(f'feat_sims shape: {tuple(feat_sims.shape)}')
print(f'per-direction stats:')
for i, n in enumerate(names):
    s = feat_sims[i]
    print(f'  {n:>16s}   max={s.max():+.4f}  min={s.min():+.4f}  mean={s.mean():+.4f}  std={s.std():.4f}')

In [ ]:
# Top-K features per direction (positive side).
top_k_pos = {}   # name -> {'idx': [...], 'sim': [...]}
for i, n in enumerate(names):
    tk = torch.topk(feat_sims[i], TOP_K, largest=True)
    top_k_pos[n] = {'idx': tk.indices.tolist(), 'sim': tk.values.tolist()}

print(f'Top-{TOP_K} feature IDs per direction:')
for n in names:
    ids = ' '.join(f'{i:6d}' for i in top_k_pos[n]['idx'][:10])
    print(f'  {n:>16s}   {ids} ...')

## 5 — Overlap between top-K feature sets (Jaccard)

For each pair of directions, fraction of top-K features they share. High Jaccard = directions decompose into overlapping features (likely related). Low Jaccard = orthogonal feature stories.

In [ ]:
n = len(names)
jaccard = np.zeros((n, n))
for i in range(n):
    a = set(top_k_pos[names[i]]['idx'])
    for j in range(n):
        b = set(top_k_pos[names[j]]['idx'])
        union = len(a | b)
        jaccard[i, j] = len(a & b) / union if union else 0.0

print(f'Jaccard overlap of top-{TOP_K} feature sets:')
print(' ' * 18 + ' '.join(f'{n:>16s}' for n in names))
for i in range(len(names)):
    row = ' '.join(f'{jaccard[i, j]:16.3f}' for j in range(len(names)))
    print(f'{names[i]:>16s}  {row}')

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(2 + 1.6 * len(names), 1 + 0.9 * len(names)))
    for ax, mat, title, vmin, vmax, cmap in [
        (axes[0], cos_matrix, 'cosine (direction-space)', -1, 1, 'RdBu_r'),
        (axes[1], jaccard,    f'Jaccard top-{TOP_K} features', 0, 1, 'YlGn'),
    ]:
        im = ax.imshow(mat, vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha='right')
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
        for i in range(len(names)):
            for j in range(len(names)):
                v = mat[i, j]
                ax.text(j, i, f'{v:+.2f}' if vmin < 0 else f'{v:.2f}', ha='center', va='center',
                        color='white' if (abs(v) > 0.5 if vmin < 0 else v > 0.5) else 'black',
                        fontsize=9)
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not available)')

## 6 — Per-direction top-features grid (with Neuronpedia URLs)

For each direction, the top-K feature IDs and their cosine to that direction. Click through Neuronpedia URLs (or run notebook 11's auto-interp cell on each direction individually) to read descriptions.

In [ ]:
NEURONPEDIA_MODEL_SLUG = 'llama3.3-70b'   # confirm on neuronpedia.org
NEURONPEDIA_SAE_SLUG   = '50-goodfire-sae-l50'  # placeholder

for n in names:
    print(f'\n=== {n} ===')
    for rank, (idx, sim) in enumerate(zip(top_k_pos[n]['idx'], top_k_pos[n]['sim'])):
        url = f'https://www.neuronpedia.org/{NEURONPEDIA_MODEL_SLUG}/{NEURONPEDIA_SAE_SLUG}/{idx}'
        print(f'  #{rank+1:2d}  feature_{idx:6d}   cos={sim:+.4f}   {url}')

## 7 — Shared vs unique features

For each pair of directions, list the features they have in common in the top-K — and the features unique to each. This is the most readable summary: which features tell the *shared* story between (e.g.) S/U and Refusal, vs which features distinguish them.

In [ ]:
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a_name, b_name = names[i], names[j]
        a = set(top_k_pos[a_name]['idx'])
        b = set(top_k_pos[b_name]['idx'])
        shared = sorted(a & b)
        only_a = sorted(a - b)
        only_b = sorted(b - a)
        print(f'\n--- {a_name}  vs  {b_name} ---')
        print(f'  shared ({len(shared):3d}): {shared[:15]}')
        print(f'  only {a_name:>14s} ({len(only_a):3d}): {only_a[:8]}')
        print(f'  only {b_name:>14s} ({len(only_b):3d}): {only_b[:8]}')

## 8 — Save outputs

Writes the cosine matrix, Jaccard matrix, per-direction top-K features, and metadata to disk so downstream analysis / plots don't re-run the SAE load.

In [ ]:
import json

OUT_DIR = Path.cwd().parent / 'exp_direction_comparison' / 'llama33_70b_l50'
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUT_DIR / 'comparison.npz',
    names=np.array(names),
    cos_matrix=cos_matrix,
    jaccard=jaccard,
    feat_sims=feat_sims.numpy(),
)

report = {
    'sae_repo': SAE_REPO_ID,
    'sae_layer': SAE_LAYER,
    'd_model': int(d_model),
    'd_sae': int(d_sae),
    'top_k': int(TOP_K),
    'directions': names,
    'pairwise_cosine': {
        f'{names[i]}__vs__{names[j]}': float(cos_matrix[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
    },
    'pairwise_jaccard_topk': {
        f'{names[i]}__vs__{names[j]}': float(jaccard[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
    },
    'top_k_features': {n: top_k_pos[n] for n in names},
}
with open(OUT_DIR / 'report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f'wrote {OUT_DIR / "comparison.npz"}')
print(f'wrote {OUT_DIR / "report.json"}')